In [6]:
import pandas as pd
import glob
import os
from concurrent.futures import ProcessPoolExecutor

In [7]:
# Definir la ruta de la carpeta que contiene los archivos
#carpeta_eto = './ETO'
carpeta_eto = './Datos/ETO_Mensual_2022_2023'

In [8]:
# Crear una lista para almacenar los DataFrames de cada archivo
dataframes = []

# Leer todos los archivos en la carpeta con el patrón "Reporte Evapotranspiracion *-22.xlsx" y "*-23.xlsx"
archivos = glob.glob(os.path.join(carpeta_eto, 'Reporte Evapotranspiracion *-*.xlsx'))

In [9]:
# Función para procesar cada archivo individualmente
def procesar_archivo(archivo):
    # Extraer el año del nombre del archivo
    nombre_archivo = os.path.basename(archivo)
    año = '20' + nombre_archivo.split('-')[-1].replace('.xlsx', '')

    # Leer el archivo en un DataFrame usando el motor openpyxl
    df = pd.read_excel(archivo, engine='openpyxl')

    # Convertir la columna de fecha para incluir el año
    df['FECHA'] = pd.to_datetime(df['FECHA'] + '/' + año, format='%d/%m/%Y')

    # Seleccionar solo las columnas necesarias
    df = df[['REFERENCIA', 'FECHA', 'ETO']]
    
    return df

# Usar ProcessPoolExecutor para procesar los archivos en paralelo
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    # Procesar los archivos en paralelo y recoger los resultados en una lista
    dataframes = list(executor.map(procesar_archivo, archivos))

# Concatenar todos los DataFrames en uno solo después de la paralelización
df_combinado = pd.concat(dataframes)

# Eliminar duplicados, conservando el primer registro encontrado para cada combinación de 'REFERENCIA' y 'FECHA'
df_combinado = df_combinado.drop_duplicates(subset=['REFERENCIA', 'FECHA'], keep='first')

# Ordenar por referencia y fecha
df_combinado = df_combinado.sort_values(by=['REFERENCIA', 'FECHA']).reset_index(drop=True)

# Guardar el DataFrame combinado en un archivo Excel
df_combinado.to_excel('./Datos/ETo_REMAS_Etchojoa_Cajeme_2022-2023.xlsx', index=False)
print("Archivo combinado creado como 'ETo_REMAS_Etchojoa_Cajeme_2022-2023.xlsx'")

Archivo combinado creado como 'ETo_REMAS_Etchojoa_Cajeme_2022-2023.xlsx'
